In [2]:
## Load h2o and prerequisites libraries

import numpy as np
import pandas as pd
import warnings
import json
import psycopg2
import arrow
import pyarrow as pa
import pyarrow.parquet as pq



In [2]:


# Replace these values with your actual database information
host = "lwanalytics-dev.cwrjruraj02j.us-east-1.rds.amazonaws.com"          # e.g., localhost or your database IP address
database = ""
user = ""
password = ""
port = "5432"           # e.g., 5432

# Creating the connection string
connection_string = f"host={host} dbname={database} user={user} password={password} port={port}"

# Connecting to the PostgreSQL database
try:
    connection = psycopg2.connect(connection_string)
    print("Connected to the database!")

    # Further actions with the database can be performed here

except psycopg2.Error as e:
    print("Unable to connect to the database.")
    print(e)
finally:
    # Close the connection when done
    if connection:
        connection.close()
        print("Connection closed.")

Connected to the database!
Connection closed.


In [3]:
connection = psycopg2.connect(connection_string)

In [4]:
cursor = connection.cursor()
column_names = ['audit_table_name', 'audit_action', 'audit_timestamp','audit_type','audit_counter','trans_order','table_name','table_key','reason','record_signed','user_name','user_role','parent_table','parent_key','batch_name','col_num','document_number','experiment_number','row_num','workbook_number','worksheet','formulation','investigation','order_num','project_name','name','report_number','sample_number','location_number','t_mlar','t_mlar_severity','trans_string']
sql_command = "select 'sample_audit_log',action,audit_timestamp,audit_type,counter,trans_order,table_name,table_key,reason,record_signed,user_name,user_role,'','','',NULL,NULL,NULL,NULL,NULL,'','','','','','','',sample_number,'',t_mlar,t_mlar_severity,trans_string from SAMPLE_AUDIT_DECRYPT"
cursor.execute(sql_command)
queryFetch = cursor.fetchall()
dataFrame = pd.DataFrame.from_records(queryFetch, columns = column_names)

In [5]:
def loads_json(x):
    try:
        return json.loads(x) if pd.notna(x) else pd.Series()
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Problematic JSON string: {x}")
        return pd.Series()


dataFrame['trans_string'] = dataFrame['trans_string'].apply(loads_json)
jsonData = pd.json_normalize(dataFrame['trans_string'])

Error decoding JSON: Invalid control character at: line 1 column 119 (char 118)
Problematic JSON string: {"TEST_NUMBER":"7352","ANALYSIS":"TITER_STRENGTH","REPLICATE_COUNT":"1","TEST_COMMENT":"Capsid Record No. 30Sep2022GM1
Vg Titer 1 Record No. 03OCT2022AN1
Vg Titer 2 Record No. 04Oct2022GM1
Vg Titer 3 Record No. 05OCT2022CA1","CHANGED_ON":"2022-12-01 14:37:34"}
Error decoding JSON: Invalid control character at: line 1 column 112 (char 111)
Problematic JSON string: {"TEST_NUMBER":"1368","ANALYSIS":"BACPAK","REPLICATE_COUNT":"1","TEST_COMMENT":"QC Sample number: QCS-2022-0102
Test Record: A10007 - A10015","CHANGED_ON":"2023-01-17 14:48:38"}
Error decoding JSON: Invalid control character at: line 1 column 122 (char 121)
Problematic JSON string: {"TEST_NUMBER":"3569","ANALYSIS":"TITER_STRENGTH","REPLICATE_COUNT":"1","TEST_COMMENT":"Capsid Test Record: 01Sep2022CVG1
Vg Titer Test Record: 22AUG2022SC1","CHANGED_ON":"2023-01-17 15:11:40"}
Error decoding JSON: Invalid control character at: l

In [6]:
jsonData = jsonData.add_suffix('_norm')

In [7]:
dataFrame = pd.concat([dataFrame,jsonData],axis=1)

In [8]:
dataFrame = dataFrame.drop(columns=['trans_string'])


In [9]:
dataFrame['customer'] = "tenaya"

In [10]:
table = pa.Table.from_pandas(dataFrame)

In [11]:
pq.write_table(table,'C:\MLAR_DATA\T0001/MLAR_RAW_tenaya.parquet')

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\dan.larson\AppData\Local\Temp\2\ipykernel_10356\4170909979.py:1: SyntaxWarning: invalid escape sequence '\M'
  pq.write_table(table,'C:\MLAR_DATA\T0001/MLAR_RAW_tenaya.parquet')


In [4]:
import duckdb as db
dat = db.sql(f"select * from read_parquet('C:/MLAR_DATA/T0001/MLAR_RAW_seres.parquet')").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
dat = dat.replace(np.nan,-999).replace("▄",",",regex=True)

In [7]:
table = pa.Table.from_pandas(dat)


ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column audit_action with type object')

In [ ]:
pq.write_table(table,'C:\MLAR_DATA\T0001/MLAR_RAW_seres.parquet')